In [ ]:
import numpy as np
import math
import os
from pathlib import Path
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd
import pyvista as pv

In [ ]:
# Historical preview cell. The surfaces are generated later in this notebook,
# so opening them here made Run All fail in a clean directory.
print("phantom surfaces are generated below")


In [ ]:
# A historical display-only cell that opened restricted study files was removed.
# None of its variables was used downstream. The phantom below is parametric and
# has no patient input at any stage.


In [ ]:
def align_ellipsoid_to_vector(ellipsoid, target_vec, pos):
    # Normalize target vector
    target_vec = target_vec / np.linalg.norm(target_vec)
    # Default ellipsoid long axis is along Z
    z_axis = np.array([0, 0, 1])
    # Axis of rotation
    rot_axis = np.cross(z_axis, target_vec)
    rot_axis_norm = np.linalg.norm(rot_axis)
    if rot_axis_norm < 1e-8:
        rotated = ellipsoid
    else:
        rot_axis /= rot_axis_norm
        # Angle between z_axis and target_vec
        angle = np.degrees(np.arccos(np.dot(z_axis, target_vec)))
        rotated = ellipsoid.rotate_vector(rot_axis, angle, point=(0,0,0), inplace=False)
    # Translate after rotation
    rotated = rotated.translate(pos)
    return rotated

def create_body(radius, control_points, ball_radius=[10,10,12], rotation=0, n_points=100):
    centerline_spline_temp = pv.Spline(control_points, n_points)
    centerline = centerline_spline_temp.points
    n_radial = 60  
    centerline_spline = pv.Spline(centerline, n_points)

    tube = centerline_spline.tube(radius=radius, n_sides=n_radial)
    center_point = centerline[0]
    center_normal = centerline[1] - centerline[0]
    center_normal_normalized = center_normal / np.linalg.norm(center_normal)

    if abs(center_normal_normalized[0]) < 0.9:
        perp1 = np.cross(center_normal_normalized, np.array([1, 0, 0]))
    else:
        perp1 = np.cross(center_normal_normalized, np.array([0, 1, 0]))
    perp1 /= np.linalg.norm(perp1)
    perp2 = np.cross(center_normal_normalized, perp1)
    perp2 /= np.linalg.norm(perp2)

    angle_spacing = 2 * np.pi / 3
    ball_positions = []
    for i in range(3):
        angle = i * angle_spacing + rotation
        offset = 12 * (np.cos(angle) * perp1 + np.sin(angle) * perp2)
        ball_positions.append(center_point + offset)

    balls = []
    for i, pos in enumerate(ball_positions):
        if i == 0:
            ellipsoid = pv.ParametricEllipsoid(
                xradius=ball_radius[0],
                yradius=ball_radius[1],
                zradius=ball_radius[2]+5
            ).triangulate()
        else:
            ellipsoid = pv.ParametricEllipsoid(
                xradius=ball_radius[0],
                yradius=ball_radius[1],
                zradius=ball_radius[2]
            ).triangulate()

        # Align and then translate
        ellipsoid = align_ellipsoid_to_vector(ellipsoid, center_normal_normalized, pos)
        balls.append(ellipsoid)

    combined = tube.merge(balls[0]).merge(balls[1]).merge(balls[2])
    return combined, centerline_spline

In [ ]:
# Define 3 control points for the centerline
# Format: [x, y, z] in mm
control_points = np.array([
    [0.0, 0.0, 0.0],      # Start point (inlet)
    [20.0, 0.0, 35.0],    # Middle point (curve peak)
    [40.0, 0.0, 45.0]    # End point (outlet)
])

ata_orig,centerline=create_body(radius=15,rotation=0 ,control_points=control_points)

In [ ]:

control_points = np.array([
    [0, 15.0, 0],      # Start point (inlet)
    [20.0, 3.0, 35.0],    # Middle point (curve peak)
    [40.0, 0.0, 45.0]
])
ata_def1,centerline_def1 =create_body(radius=16,rotation=-0.22 ,control_points=control_points)


In [ ]:
control_points = np.array([
    [-10.0, -10.0, 0],      # Start point (inlet)
    [20.0, 3.0, 35.0],    # Middle point (curve peak)
    [40.0, 0.0, 45.0]     # End point (outlet)
])
ata_def2,centerline_def2 =create_body(radius=17,rotation=0.22 ,control_points=control_points)

In [ ]:
control_points = np.array([
    [10.0, 10.0, 0],      # Start point (inlet)
    [20.0, 3.0, 35.0],    # Middle point (curve peak)
    [40.0, 0.0, 45.0]     # End point (outlet)
])
ata_def3,centerline_def3 =create_body(radius=18,rotation=-0.52,control_points=control_points)

In [ ]:
plotter = pv.Plotter(notebook=True)
plotter.add_mesh(ata_orig, color='lightblue', opacity=1, label='Original ATA')
plotter.add_mesh(centerline, color='blue', line_width=5, label='Original Centerline')

plotter.show()

In [ ]:
plotter = pv.Plotter(notebook=False)
plotter.add_mesh(ata_orig, color='lightblue', opacity=1)
plotter.add_mesh(ata_def1, color='green', opacity=0.5)
plotter.add_mesh(ata_def2, color='red', opacity=0.5)
plotter.add_mesh(ata_def3, color='blue', opacity=0.5)
plotter.show()

In [ ]:
centerline_point_inlet=25
centerline_point_arc=92

arc_center=centerline.points[centerline_point_arc]
arc_normal=centerline.points[centerline_point_arc+1]-centerline.points[centerline_point_arc]
ata_orig.clip(origin=arc_center, normal=arc_normal,inplace=True)
ata_def1.clip(origin=arc_center, normal=arc_normal,inplace=True)
ata_def2.clip(origin=arc_center, normal=arc_normal,inplace=True)    
ata_def3.clip(origin=arc_center, normal=arc_normal,inplace=True)


ata_orig.compute_normals(inplace=True).fill_holes(1000, inplace=True).fill_holes(10,inplace=True)
ata_def1.compute_normals(inplace=True).fill_holes(1000, inplace=True).fill_holes(10,inplace=True)
ata_def2.compute_normals(inplace=True).fill_holes(1000, inplace=True).fill_holes(10,inplace=True)
ata_def3.compute_normals(inplace=True).fill_holes(1000, inplace=True).fill_holes(10,inplace=True)



plotter = pv.Plotter(notebook=False)

plotter.add_points(centerline.points, color='blue', point_size=10, render_points_as_spheres=True)
plotter.add_mesh(ata_orig,color='cyan', opacity=0.8, label='Centerline Tube')
# plotter.add_mesh(ata_def1,color='red', opacity=0.5, label='Centerline Tube Def1')
# plotter.add_mesh(ata_def2,color='green', opacity=0.5, label='Centerline Tube Def2')
# plotter.add_mesh(ata_def3,color='blue', opacity=0.5, label='Centerline Tube Def3')



plane=pv.Plane(center=centerline.points[centerline_point_inlet], direction=centerline.points[centerline_point_inlet+1]-centerline.points[centerline_point_inlet], i_size=40, j_size=40)
plotter.add_mesh(plane)

plotter.add_axes()
plotter.add_text("PyVista Centerline", font_size=12)
plotter.show_grid()

vec1 = centerline.points[centerline_point_inlet+1]-centerline.points[centerline_point_inlet]


vec_normalized1 = vec1 / np.linalg.norm(vec1)
print("center:",centerline.points[centerline_point_inlet])
print("direction:",vec_normalized1)


plane=pv.Plane(center=centerline.points[centerline_point_arc], direction=centerline.points[centerline_point_arc+1]-centerline.points[centerline_point_arc], i_size=30, j_size=30)
# plotter.add_mesh(plane)

plotter.add_axes()
plotter.add_text("PyVista centerline_point_arc", font_size=12)
plotter.show_grid()

vec = centerline.points[centerline_point_arc+1] - centerline.points[centerline_point_arc]


vec_normalized = vec / np.linalg.norm(vec)

print("center arc:",centerline.points[centerline_point_arc])
print("direction arc:",vec_normalized)






plotter.show()

In [ ]:
# Write the control planes that notebook 2 reads.
#
# Added in depositing this material. The cell above already computes both planes
# and prints them; the delivered cuts_posit.txt holds exactly those numbers, to
# every printed digit, so the author transcribed them by hand and the notebooks
# were never joined up. This writes the same file from the same variables, in the
# format read_cuts_posit() parses.
with open("cuts_posit.txt", "w") as fh:
    fh.write("Name\tSource Point\tnormal\n")
    fh.write(f"Inlet\t{centerline.points[centerline_point_inlet]}\t{vec_normalized1}\n")
    fh.write(f"arc\t{centerline.points[centerline_point_arc]}\t{vec_normalized}\n")
print("wrote cuts_posit.txt")


In [ ]:
centerline.save('centerline.vtk')
ata_orig.save('ATAA_sint.stl')
centerline_def1.save('centerline_def1.vtk')
ata_def1.save('ATAA_sint_def1.stl')

centerline_def2.save('centerline_def2.vtk')
ata_def2.save('ATAA_sint_def2.stl')

centerline_def3.save('centerline_def3.vtk')
ata_def3.save('ATAA_sint_def3.stl')

In [ ]:
# ata2_orig=pv.read('ATAA_sint.stl')
# ata2_def1=pv.read('ATAA_sint_def1.stl')
# ata2_def2=pv.read('ATAA_sint_def2.stl')
# ata2_def3=pv.read('ATAA_sint_def3.stl')


In [ ]:
# Optional legacy volume-mesh preview. No volume mesh is generated in this
# public notebook, so there is nothing to display in a clean Run All.

In [ ]:
print("surface phantom generation complete")